In [1]:
pip install torch transformers scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

class MultiModelAIDetector(nn.Module):
    def __init__(self, transformer_name='roberta-base', tfidf_dim=1000, hidden_dim=256):
        super(MultiModelAIDetector, self).__init__()
        
        self.transformer = AutoModel.from_pretrained(transformer_name)
        transformer_out_dim = self.transformer.config.hidden_size 
       
        combined_dim = transformer_out_dim + tfidf_dim
        
        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, 1) # Outputting raw logits
        )

    def forward(self, input_ids, attention_mask, tfidf_vectors):
        
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        
        h_cls = outputs.last_hidden_state[:, 0, :] 
        
        z = torch.cat((h_cls, tfidf_vectors), dim=1)
        
        logits = self.classifier(z)
        return logits

C:\Users\athar\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# --- 1. Setup and Initialization ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained('roberta-base')
tfidf_vectorizer = TfidfVectorizer(max_features=1000, ngram_range=(1, 3)) 

# --- 2. Mock Data ---
texts = [
    "The rapid advancement of machine learning has introduced new paradigms in computing.",
    "I'm going to the store to grab some milk, want anything?"
]

# --- 3. Preprocessing (Run this BEFORE initializing the PyTorch model) ---
# Fit the vectorizer and get the actual matrix
tfidf_matrix = tfidf_vectorizer.fit_transform(texts).toarray()
tfidf_tensors = torch.tensor(tfidf_matrix, dtype=torch.float32).to(device)

# THE FIX: Dynamically grab the exact number of TF-IDF features generated
actual_tfidf_dim = tfidf_matrix.shape[1] 
print(f"Actual TF-IDF vocabulary size: {actual_tfidf_dim}")

# --- 4. Initialize the custom fusion model with the CORRECT dimension ---
model = MultiModelAIDetector(
    transformer_name='roberta-base', 
    tfidf_dim=actual_tfidf_dim  # <--- Passing the dynamic dimension here
).to(device)

# --- 5. Semantic Tokenization ---
encoded_inputs = tokenizer(
    texts, 
    padding=True, 
    truncation=True, 
    max_length=512, 
    return_tensors='pt'
).to(device)

# --- 6. Forward Pass ---
model.eval()
with torch.no_grad():
    logits = model(
        input_ids=encoded_inputs['input_ids'], 
        attention_mask=encoded_inputs['attention_mask'], 
        tfidf_vectors=tfidf_tensors
    )
    probabilities = torch.sigmoid(logits).squeeze()

# --- 7. Output ---
for i, text in enumerate(texts):
    prob = probabilities[i].item() * 100
    print(f"Text: '{text[:40]}...' | AI Probability: {prob:.2f}%")

Actual TF-IDF vocabulary size: 58


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Text: 'The rapid advancement of machine learnin...' | AI Probability: 53.60%
Text: 'I'm going to the store to grab some milk...' | AI Probability: 53.12%


In [6]:
pip install datasets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from datasets import load_dataset
from tqdm import tqdm

# ==========================================
# 1. THE CUSTOM DATASET CLASS
# ==========================================
class AIDetectionDataset(Dataset):
    def __init__(self, encodings, tfidf_sparse_matrix, labels):
        self.encodings = encodings
        self.tfidf_sparse_matrix = tfidf_sparse_matrix
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Grab the transformer tokens
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        
        # Grab the TF-IDF row, convert from sparse to dense array, then to tensor
        tfidf_row = self.tfidf_sparse_matrix[idx].toarray().squeeze()
        item['tfidf_vectors'] = torch.tensor(tfidf_row, dtype=torch.float32)
        
        # Grab the label
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float32)
        
        return item

# ==========================================
# 2. DATA PREPARATION (The Bridge)
# ==========================================
print("Loading dataset...")
hf_dataset = load_dataset("shahxeebhassan/human_vs_ai_sentences")

# THE FIX: Shuffle the dataset with a fixed seed for reproducibility
shuffled_dataset = hf_dataset['train'].shuffle(seed=42)

# Now take your training chunk
train_data = shuffled_dataset.select(range(5000))
texts = train_data['text']
labels = train_data['label'] 

# And take your testing chunk (which will now have a mix of 0s and 1s)
test_data = shuffled_dataset.select(range(5000, 6000)) 
test_texts = test_data['text']
test_labels = test_data['label']

print("Fitting TF-IDF Vectorizer...")
# We use max_features to control the size of our fusion layer
tfidf_vectorizer = TfidfVectorizer(max_features=1000, ngram_range=(1, 3))
tfidf_sparse = tfidf_vectorizer.fit_transform(texts)
actual_tfidf_dim = tfidf_sparse.shape[1]

print("Tokenizing for RoBERTa...")
tokenizer = AutoTokenizer.from_pretrained('roberta-base')
encodings = tokenizer(
    texts, 
    truncation=True, 
    padding=True, 
    max_length=256, # Keeping it shorter speeds up training 
    return_tensors=None # Keep as lists for the Dataset class to handle
)

# Create the PyTorch Dataset and DataLoader
train_dataset = AIDetectionDataset(encodings, tfidf_sparse, labels)
# Batch size of 16 or 32 is usually safe for an 8GB GPU
train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# ==========================================
# 3. INITIALIZE THE MODEL
# ==========================================
# Note: Ensure your MultiModelAIDetector class from the previous step is defined above this!
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = MultiModelAIDetector(
    transformer_name='roberta-base', 
    tfidf_dim=actual_tfidf_dim
).to(device)

# ==========================================
# 4. THE TRAINING LOOP
# ==========================================
optimizer = optim.AdamW(model.parameters(), lr=2e-5)
criterion = nn.BCEWithLogitsLoss()

def train_model(model, dataloader, epochs=3):
    model.train()
    
    for epoch in range(epochs):
        total_loss = 0
        print(f"\n--- Epoch {epoch+1}/{epochs} ---")
        
        # Wrap dataloader in tqdm for a progress bar
        progress_bar = tqdm(dataloader, desc="Training")
        
        for batch in progress_bar:
            # Move data to GPU/CPU
            b_input_ids = batch['input_ids'].to(device)
            b_masks = batch['attention_mask'].to(device)
            b_tfidf = batch['tfidf_vectors'].to(device)
            b_labels = batch['labels'].to(device)
            
            optimizer.zero_grad()
            
            # Forward pass
            logits = model(b_input_ids, b_masks, b_tfidf).squeeze()
            
            # Calculate loss
            loss = criterion(logits, b_labels)
            total_loss += loss.item()
            
            # Backward pass & optimize
            loss.backward()
            optimizer.step()
            
            # Update progress bar with current loss
            progress_bar.set_postfix({'loss': loss.item()})
            
        avg_loss = total_loss / len(dataloader)
        print(f"Average Training Loss: {avg_loss:.4f}")

# Start the training!
train_model(model, train_dataloader, epochs=3)

print("Training complete! Model is ready for inference.")

Loading dataset...
Fitting TF-IDF Vectorizer...
Tokenizing for RoBERTa...
Using device: cuda


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



--- Epoch 1/3 ---


Training: 100%|███████████████| 313/313 [01:47<00:00,  2.92it/s, loss=0.421]


Average Training Loss: 0.3692

--- Epoch 2/3 ---


Training: 100%|██████████████| 313/313 [01:32<00:00,  3.40it/s, loss=0.0279]


Average Training Loss: 0.1799

--- Epoch 3/3 ---


Training: 100%|██████████████| 313/313 [01:35<00:00,  3.28it/s, loss=0.0248]

Average Training Loss: 0.1180
Training complete! Model is ready for inference.


In [17]:
# Assuming you already loaded hf_dataset from the previous training step
# Let's grab a separate chunk of data that the model has NEVER seen
print("Preparing holdout test data...")
test_data = shuffled_dataset.select(range(5000, 6000)) 

test_texts = test_data['text']
test_labels = test_data['label']

# 1. Statistical Vectorization (Crucial: Use .transform() only!)
test_tfidf_sparse = tfidf_vectorizer.transform(test_texts)

# 2. Semantic Tokenization
test_encodings = tokenizer(
    test_texts, 
    truncation=True, 
    padding=True, 
    max_length=256, 
    return_tensors=None
)

# 3. Create Dataset and DataLoader
# We reuse the AIDetectionDataset class we defined during training
test_dataset = AIDetectionDataset(test_encodings, test_tfidf_sparse, test_labels)

# We don't need to shuffle the test data
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# ==========================================
# RUN THE EVALUATION
# ==========================================
true_labels, predicted_labels = evaluate_model(model, test_dataloader, device)

Preparing holdout test data...
Starting evaluation...

FINAL EVALUATION METRICS
              precision    recall  f1-score   support

   Human (0)     0.9630    0.8245    0.8884       473
      AI (1)     0.8605    0.9715    0.9127       527

    accuracy                         0.9020      1000
   macro avg     0.9117    0.8980    0.9005      1000
weighted avg     0.9090    0.9020    0.9012      1000



In [18]:
import torch
from sklearn.metrics import classification_report, confusion_matrix

def evaluate_model(model, test_dataloader, device):
    print("Starting evaluation...")
    # Set model to evaluation mode (disables dropout layers, etc.)
    model.eval()
    
    all_preds = []
    all_labels = []
    
    # Disable gradient calculation for inference
    with torch.no_grad():
        for batch in test_dataloader:
            # 1. Move data to device
            b_input_ids = batch['input_ids'].to(device)
            b_masks = batch['attention_mask'].to(device)
            b_tfidf = batch['tfidf_vectors'].to(device)
            b_labels = batch['labels'].to(device)
            
            # 2. Forward pass
            logits = model(b_input_ids, b_masks, b_tfidf).squeeze()
            
            # 3. Convert logits to probabilities (Sigmoid) and then to binary classes (0 or 1)
            # A threshold of 0.5 means if probability > 50%, predict AI (1)
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).int()
            
            # 4. Store the results
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(b_labels.cpu().numpy())
            
    # 5. Print the final metrics
    print("\n" + "="*50)
    print("FINAL EVALUATION METRICS")
    print("="*50)
    
    # target_names assumes 0 is Human Text, 1 is AI Text
    report = classification_report(
        all_labels, 
        all_preds, 
        target_names=['Human (0)', 'AI (1)'],
        digits=4 # Show 4 decimal places for better granularity
    )
    print(report)
    
    return all_labels, all_preds